# Week 10 · Day 2 (PyTorch) — Comparing CNN Architectures on a Real Task (Scene Recognition)

Same comparison as the TensorFlow class notebook, in **PyTorch** — our main framework.

- Dataset: **Intel Image Classification** — ~14k train / ~3k test color images, **6 scene classes**.
- We compare **5 architectures** from the CNN-history talk: **InceptionV3, ResNet50, DenseNet121, MobileNetV2, EfficientNetB0**.
- Method: **feature extraction** — freeze each pretrained backbone, train a small head.
- Compare on: **accuracy · speed · model size**. Then **fine-tune the winner**.

> **Kaggle GPU:** Settings → Accelerator → GPU, then add the Intel Image Classification dataset via Add Input.

> Organised sequentially: train & evaluate one architecture at a time, then compare at the end.

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Load the data with `ImageFolder`

- Intel Scenes is already split into `seg_train` and `seg_test`, each with one folder per class.
- On Kaggle the folders are **nested one level** (`seg_train/seg_train/<class>/`) — adjust if yours differs.
- `torchvision.datasets.ImageFolder` reads class folders directly — the built-in version of our custom Dataset.
- We resize to **224×224** and normalize with **ImageNet stats** (what the pretrained models expect).

In [ ]:
TRAIN_DIR = "/kaggle/input/intel-image-classification/seg_train/seg_train"
TEST_DIR  = "/kaggle/input/intel-image-classification/seg_test/seg_test"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_data = datasets.ImageFolder(TRAIN_DIR, transform=tf)
test_data  = datasets.ImageFolder(TEST_DIR,  transform=tf)

class_names = train_data.classes
n_classes = len(class_names)
print("classes:", class_names)
print("train:", len(train_data), " test:", len(test_data))

In [ ]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True,
                          num_workers=2, pin_memory=(device.type == "cuda"))
test_loader  = DataLoader(test_data,  batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=(device.type == "cuda"))

# test labels once, for metrics
y_test = np.array(test_data.targets)

In [ ]:
# look at a few images (undo normalization for display)
def denorm(t):
    x = t.numpy().transpose(1, 2, 0)
    return np.clip(x * IMAGENET_STD + IMAGENET_MEAN, 0, 1)

imgs, labels = next(iter(train_loader))
plt.figure(figsize=(14, 4))
for i in range(12):
    plt.subplot(2, 6, i + 1)
    plt.imshow(denorm(imgs[i]))
    plt.title(class_names[labels[i]], fontsize=9); plt.axis("off")
plt.suptitle("Intel scenes — 6 classes")
plt.tight_layout(); plt.show()

## 2. A helper to build, train & evaluate one model

- Each backbone loads pretrained (ImageNet), gets **frozen**, and its **head is replaced** with a new `Linear` for 6 classes.
- The head lives in a different attribute per architecture (`fc`, `classifier`, ...), so a small `build_model` handles each.
- One `run_architecture` function trains the head, evaluates, and records **accuracy, macro-F1, train time, infer time, params.**

In [ ]:
def build_model(name):
    """Load a pretrained backbone, freeze it, replace its head for n_classes."""
    if name == "InceptionV3":
        m = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1)
        m.aux_logits = False; m.AuxLogits = None      # ignore the auxiliary head
        for p in m.parameters(): p.requires_grad = False
        m.fc = nn.Linear(m.fc.in_features, n_classes)
    elif name == "ResNet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        for p in m.parameters(): p.requires_grad = False
        m.fc = nn.Linear(m.fc.in_features, n_classes)
    elif name == "DenseNet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        for p in m.parameters(): p.requires_grad = False
        m.classifier = nn.Linear(m.classifier.in_features, n_classes)
    elif name == "MobileNetV2":
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        for p in m.parameters(): p.requires_grad = False
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
    elif name == "EfficientNetB0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        for p in m.parameters(): p.requires_grad = False
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
    return m.to(device)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
results = []

@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds = []
    for xb, _ in loader:
        preds.append(model(xb.to(device)).argmax(1).cpu())
    return torch.cat(preds).numpy()

def run_architecture(name, epochs=4):
    model = build_model(name)
    # only the head's params need training
    head_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(head_params, lr=1e-3)

    t0 = time.time()
    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb), yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    train_time = time.time() - t0

    t1 = time.time()
    preds = predict(model, test_loader)
    infer_time = time.time() - t1

    acc = (preds == y_test).mean()
    macro_f1 = f1_score(y_test, preds, average="macro")
    params = sum(p.numel() for p in model.parameters())
    results.append({"model": name, "accuracy": acc, "macro_f1": macro_f1,
                    "train_s": train_time, "infer_s": infer_time, "params": params})
    print(f"{name}:  acc {acc:.2%} | macro-F1 {macro_f1:.3f} | train {train_time:.0f}s | infer {infer_time:.1f}s | params {params/1e6:.1f}M")
    return model

## 3. Train & evaluate each architecture, one at a time

### 3a. InceptionV3  — *multi-scale filters (2014)*

In [ ]:
_ = run_architecture("InceptionV3")

### 3b. ResNet50  — *skip connections (2015)*

In [ ]:
_ = run_architecture("ResNet50")

### 3c. DenseNet121  — *dense feature reuse (2016)*

In [ ]:
_ = run_architecture("DenseNet121")

### 3d. MobileNetV2  — *lightweight, for phones (2017)*

In [ ]:
_ = run_architecture("MobileNetV2")

### 3e. EfficientNetB0  — *compound scaling (2019)*

In [ ]:
_ = run_architecture("EfficientNetB0")

## 4. The results table

In [ ]:
res = pd.DataFrame(results).sort_values("macro_f1", ascending=False).reset_index(drop=True)
d = res.copy()
d["accuracy"] = (d["accuracy"] * 100).round(1).astype(str) + "%"
d["macro_f1"] = d["macro_f1"].round(3)
d["train_s"] = d["train_s"].round(0).astype(int)
d["infer_s"] = d["infer_s"].round(1)
d["params"] = (d["params"] / 1e6).round(1).astype(str) + "M"
d.columns = ["Model", "Accuracy", "Macro-F1", "Train (s)", "Infer (s)", "Params"]
print(d.to_string(index=False))

## 5. Visualize the trade-offs

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

xr = np.arange(len(res))
ax1.bar(xr, res["accuracy"] * 100, color="steelblue")
ax1.set_xticks(xr); ax1.set_xticklabels(res["model"], rotation=30, ha="right")
ax1.set_ylabel("test accuracy (%)"); ax1.set_title("Accuracy by architecture"); ax1.grid(alpha=0.3)
for i, v in enumerate(res["accuracy"] * 100):
    ax1.text(i, v + 0.5, f"{v:.1f}", ha="center", fontsize=9)

ax2.scatter(res["params"] / 1e6, res["accuracy"] * 100, s=140, color="green")
for _, r in res.iterrows():
    ax2.annotate(r["model"], (r["params"] / 1e6, r["accuracy"] * 100),
                 textcoords="offset points", xytext=(6, 4), fontsize=9)
ax2.set_xlabel("parameters (millions)"); ax2.set_ylabel("test accuracy (%)")
ax2.set_title("Accuracy vs model size"); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**How to read this (the field skill):**
- The most *accurate* model isn't always the right choice. A phone app wants **MobileNet** (tiny, fast); a server that just needs the best score might take the heavier net.
- Look at **accuracy per million parameters** — the small models (MobileNet, EfficientNet) are often astonishingly efficient.
- This trade-off *is* the job: there's no single "best" — only best-for-a-purpose.

## 6. Fine-tune the winner

- Take the top model by macro-F1.
- **Unfreeze** the backbone and train a little more with a **very small learning rate** (100× smaller) — gently adapting ImageNet features to scenes.

In [ ]:
winner = res.iloc[0]["model"]
print("winner by macro-F1:", winner)

model = build_model(winner)
# train the head first
opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-3)
for _ in range(4):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()

# --- fine-tune: unfreeze everything, tiny learning rate ---
for p in model.parameters():
    p.requires_grad = True
opt = torch.optim.Adam(model.parameters(), lr=1e-5)   # 100x smaller
for epoch in range(3):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"fine-tune epoch {epoch+1}/3 done")

preds = predict(model, test_loader)
print(f"\n{winner} after fine-tuning:  acc {(preds==y_test).mean():.2%} | macro-F1 {f1_score(y_test,preds,average='macro'):.3f}")

In [ ]:
cm = confusion_matrix(y_test, preds)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
plt.title(f"{winner} (fine-tuned) — scene classification")
plt.tight_layout(); plt.show()

print(classification_report(y_test, preds, target_names=class_names, digits=3))

- The confusion matrix shows which scenes get mixed up — often visually similar pairs (glacier ↔ mountain, sea ↔ glacier), the same ones a person might hesitate on.
- Per-class recall tells you where the model is weak, class by class.

## Your turn (solo task) ✍️

Pick at least two:
1. **Fine-tune a different model** from the table and compare it to the winner.
2. **Train the top two longer** (more epochs) — does the ranking change?
3. **Add data augmentation** (`transforms.RandomHorizontalFlip`, `RandomRotation`) to the train transform — does accuracy improve?
4. **Save your best model** with `torch.save(model.state_dict(), "scene_model.pt")` (needed for deployment later).

In [ ]:
# ===== YOUR EXPERIMENTS HERE =====



## Where CNNs are used in the real world

Scene recognition is one small corner of what convolutional networks do in production. The same "extract visual features → decide" machinery powers a huge range of applications:

### 🖼️ Image classification & organization
- **Photo apps** — auto-tagging and grouping photos ("beaches", "mountains", "documents"), the direct cousin of today's scene task.
- **Content moderation** — flagging unsafe or unwanted images at scale.
- **Visual search** — "find products that look like this."

### 🗺️ Location & mapping
- **Geo-tagging** — inferring where a photo was taken from its scene content.
- **Satellite & aerial imagery** — land-use mapping, deforestation and crop monitoring, disaster response.

### 🚗 Autonomous systems
- **Self-driving cars & drones** — recognizing road, sky, obstacles, and lanes for navigation.
- **Robotics** — letting a robot understand and move through its surroundings.

### 🏥 Medicine & science
- **Medical imaging** — detecting disease in X-rays, CT scans, dermatoscopy, and pathology slides.
- **Microscopy & astronomy** — classifying cells, particles, and galaxies.

### 🏭 Industry & security
- **Manufacturing** — spotting defects on production lines (visual quality control).
- **Agriculture** — identifying crop disease and weeds from leaf images.
- **Security & surveillance** — detection and recognition in camera feeds.

> Every one of these starts the same way you did today: a pretrained CNN backbone that already "sees," adapted with a small amount of task-specific training. **That is why transfer learning is the workhorse of applied computer vision.**

## Summary

- **Real task, clean data:** classifying 6 natural-scene types from the Intel dataset with `ImageFolder`.
- **Architecture comparison, one at a time:** InceptionV3, ResNet50, DenseNet121, MobileNetV2, EfficientNetB0 — each trained and scored, then tabulated.
- **Three axes that matter in the field:** accuracy, speed, and model size. No single winner.
- **Fine-tuning the winner** (unfreeze + tiny LR) gave a final boost.
- The real skill isn't "pick the best model" — it's picking the **right model for the constraints** (device, latency, accuracy needs).

> **PyTorch note:** each architecture keeps its classifier head in a different attribute (`fc` for ResNet/Inception, `classifier` for DenseNet/MobileNet/EfficientNet) — that's the one fiddly bit when swapping backbones.

**Tomorrow:** object detection — not just *what* is in an image, but *where*.